# All orchestrations — the one example

One registry of tools, one tiny tabular dataset, every grid-model shape verb.
This notebook is a thin view over [`pipeline.py`](pipeline.py); the asserted
version is `tests/integration/test_all_orchestrations.py`.

Uses the **grid model** — `from simple_steps_core import grid`.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
import pipeline

wf = pipeline.run()
wf

<Workflow 20 step(s): ['readings', 'records', 'scored', 'kept', 'head2', 'ranked', 'unique_city', 'picked', 'dropped', 'renamed', 'bucketed', 'per_city', 'n_rows', 'sum_n', 'bursts', 'coords', 'spread', 'wide', 'grid_search', 'robust']>

## Every step, staged description

`describe()` says what each step is and how many cells it holds.

In [2]:
import pandas as pd
pd.DataFrame(
    [(sid, step.describe()) for sid, step in wf.steps.items()],
    columns=['step', 'describe'],
)

,step,describe
0,readings,source identity · 4 cells
1,records,source identity · 2 cells
2,scored,map scale · 4 cells
3,kept,filter is_big · 3 cells
4,head2,slice identity · 2 cells
5,ranked,sort identity · 4 cells
6,unique_city,distinct identity · 3 cells
7,picked,select identity · 4 cells
8,dropped,drop identity · 4 cells
9,renamed,rename identity · 4 cells


## map — input columns carried through, plus a payload

`Output.view()` joins the payload grid with its per-row ledger.

In [3]:
wf.step('scored').output.view()

,city,n,score,status,error,attempts,seconds,unit
0,SF,1,10,completed,None,1,0.000002,None
1,NYC,2,20,completed,None,1,0.000001,None
2,SF,3,30,completed,None,1,0.000001,None
3,LA,2,20,completed,None,1,0.000001,None


## group → collapse — the stratified summary

`group` marks each row with a key (no nesting); `collapse(by=...)` reduces
within each stratum.

In [4]:
wf.step('per_city').output.data

,bucket,value
0,SF,4
1,NYC,2
2,LA,2


## expand → widen — longer, then wider

`expand` turns each row's list into rows; `widen` spreads each record cell
into columns.

In [5]:
wf.step('spread').output.data

,city,n,value,axis,val
0,SF,1,"{'axis': 'x', 'val': 1}",x,1
1,SF,1,"{'axis': 'y', 'val': 2}",y,2
2,NYC,2,"{'axis': 'x', 'val': 2}",x,2
3,NYC,2,"{'axis': 'y', 'val': 4}",y,4
4,SF,3,"{'axis': 'x', 'val': 3}",x,3
5,SF,3,"{'axis': 'y', 'val': 6}",y,6
6,LA,2,"{'axis': 'x', 'val': 2}",x,2
7,LA,2,"{'axis': 'y', 'val': 4}",y,4


## sweep — a grid from a cross product of parameters

In [6]:
wf.step('grid_search').output.data

,model,window,value
0,a,7,a:7
1,a,30,a:30
2,b,7,b:7
3,b,30,b:30


## Failure — the ledger keeps every unit

`risky` raises on `n == 2`; those rows fail and the rest complete. `.failed`
is the exact re-drive set.

In [7]:
pipeline.failure().step('out').output.failed

,status,error,attempts,seconds,unit
1,failed,ValueError: twos are not allowed,1,0.000005,None
3,failed,ValueError: twos are not allowed,1,0.000005,None


## Staging & validation — before anything runs

In [8]:
pipeline.validation().validate()

{'readings': (),
 'scored': (),
 'bad_arg': ("scale() takes no argument 'nope'; it accepts weight",),
 'dangling': ("map over 'nowhere', which is not an earlier step",)}

## Serialization — full-session round-trip

In [9]:
original, reloaded = pipeline.roundtrip()
reloaded.step('scored').output.values == original.step('scored').output.values

True

## Verb inference — pass a reference, the shape is chosen

Passing a step reference into a tool (with no bracket verb) wires the step and
**infers** the shape verb: `map` for an ordinary tool, `filter` for a `-> bool`,
`collapse` for an `(acc, x)` reducer, `expand` for a `-> list`. The verb is
stored concretely, so it shows up selected and can be changed.

In [ ]:
pd.DataFrame(
    [(sid, wf.step(sid).operation.to_dict()['modifiers'][0]['kind'])
     for sid in ('auto_map', 'auto_filter', 'auto_collapse', 'auto_expand')],
    columns=['step', 'inferred verb'],
)